# 03 — Baseline Classifier

**Question:** how well can we classify electronics products with a *classical* pipeline (HOG features + linear model)?  This sets the floor the deep model must beat.

**Inputs:** `data/processed/products.parquet`

**Outputs:** `docs/figures/03_baseline_confusion.png`, printed accuracy + F1.

## Setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
FIG = Path('../docs/figures'); FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet('../data/processed/products.parquet')
print(df.shape, df.label.nunique())

## Extract HOG features

HOG (histogram of oriented gradients) is a classic 2005-era descriptor.  Small, fast, but blind to colour and texture.

In [ ]:
def hog_feat(path, size=128):
    img = Image.open(path).convert('L').resize((size, size))
    return hog(np.asarray(img), pixels_per_cell=(16,16), cells_per_block=(2,2))

X = np.stack([hog_feat(p) for p in df.path])
y = df.label.values
print('X shape:', X.shape)

## Train / test split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

## Logistic regression

In [ ]:
lr = LogisticRegression(max_iter=2000, n_jobs=-1)
lr.fit(X_tr, y_tr)
print(classification_report(y_te, lr.predict(X_te)))

## Linear SVM

In [ ]:
svm = LinearSVC()
svm.fit(X_tr, y_tr)
print(classification_report(y_te, svm.predict(X_te)))

## Confusion matrix (best baseline)

In [ ]:
best = lr if lr.score(X_te, y_te) >= svm.score(X_te, y_te) else svm
cm = confusion_matrix(y_te, best.predict(X_te), labels=sorted(np.unique(y)))
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm, display_labels=sorted(np.unique(y))).plot(ax=ax, cmap='Purples', colorbar=False, xticks_rotation=45)
plt.tight_layout(); plt.savefig(FIG/'03_baseline_confusion.png', dpi=140)

## Findings

- **Best baseline accuracy:** ~30–40% (expected — HOG collapses colour).
- **Confusion pattern:** _(fill after run — laptops vs monitors is the usual muddle)_.

This is the **floor** the ResNet50 transfer-learning model in `04_cnn_transfer_learning.ipynb` needs to blow past.